<h5>Project Pipeline</h5>
<li>Project Title : <strong>Chatbot</strong></li>
<li>Dataset : <strong>from Kaggle</strong></li>
<li>Data Preprocessing : <strong>Lemmatization, Tokenization, Vector, etc</strong></li>
<li>Model Training : <strong>Transformer from scratch</strong></li>
<li>Evaluation : <strong>BLEU</strong></li>
<li>Model Saving : <strong>Using PyTorch</strong></li>
<li>Testing : <strong>Basic</strong></li>
<li>API : <strong>FastAPI</strong></li>
<li>UI : <strong>Streamlit</strong></li>

In [32]:
#importing libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from collections import Counter

import pandas as pd
import math
import re

In [33]:
#loading dataset
df = pd.read_csv("dataset/Conversation.csv")
df.head()

,Unnamed: 0,question,answer
0,0,"hi, how are you doing?",i'm fine. how about yourself?
1,1,i'm fine. how about yourself?,i'm pretty good. thanks for asking.
2,2,i'm pretty good. thanks for asking.,no problem. so how have you been?
3,3,no problem. so how have you been?,i've been great. what about you?
4,4,i've been great. what about you?,i've been good. i'm in school right now.


In [34]:
#preprocess
pattern = '[^a-zA-Z0-9]'

def clean_data(data):
    return re.sub(pattern, ' ', data)

df['question'] = df["question"].apply(clean_data)
df['answer'] = df["answer"].apply(clean_data)

def format_data(data):
    text = f"<SOS> {data} <EOS>"
    return text

def join_data(data):
    text = f"{data['question']} {data['answer']}"
    return text

df["text"] = df.apply(join_data, axis=1)
df.head()

,Unnamed: 0,question,answer,text
0,0,hi how are you doing,i m fine how about yourself,hi how are you doing i m fine how about you...
1,1,i m fine how about yourself,i m pretty good thanks for asking,i m fine how about yourself i m pretty good ...
2,2,i m pretty good thanks for asking,no problem so how have you been,i m pretty good thanks for asking no problem...
3,3,no problem so how have you been,i ve been great what about you,no problem so how have you been i ve been gr...
4,4,i ve been great what about you,i ve been good i m in school right now,i ve been great what about you i ve been goo...


In [35]:
vocab = {}

def build_vocab(data, vocab_size=5000):
    words = Counter()
    for sentence in data:
        words.update(re.findall(r'\w+', sentence))

    vocab = {word: i+4 for i, (word,_) in enumerate(words.most_common(vocab_size-4)) }
    vocab["<UNK>"] = 1
    vocab["<PAD>"] = 0
    vocab["<SOS>"] = 2
    vocab["<EOS>"] = 3
    return vocab

vocab = build_vocab(df["text"])
len(vocab)

2459

In [36]:
def tokenize(data):
    word_to_idx = []
    words = re.findall(r'\w+|<\w+>',data)
    # print(words)
    for word in words:
        if word in vocab:
            # print(word)
            word_to_idx.append(vocab[word])
        else:
            word_to_idx.append(vocab["<UNK>"])
    # print(word_to_idx)
    return word_to_idx

test = df['text'].apply(tokenize)
test.head()

0      [1499, 41, 20, 5, 176, 4, 34, 604, 41, 37, 552]
1    [4, 34, 604, 41, 37, 552, 4, 34, 160, 48, 244,...
2    [4, 34, 160, 48, 244, 29, 486, 33, 172, 25, 41...
3    [33, 172, 25, 41, 18, 5, 101, 4, 71, 101, 105,...
4    [4, 71, 101, 105, 13, 37, 5, 4, 71, 101, 48, 4...
Name: text, dtype: object

In [37]:
txt = "<UNK> hi how are you doing <PAD>"
tokenize(txt)

[1, 1499, 41, 20, 5, 176, 0]

In [38]:
#formatting
df['question'] = df['question'].apply(format_data)
df['answer'] = df['answer'].apply(format_data)
df = df.drop(columns=['text'])
df.head()

,Unnamed: 0,question,answer
0,0,<SOS> hi how are you doing <EOS>,<SOS> i m fine how about yourself <EOS>
1,1,<SOS> i m fine how about yourself <EOS>,<SOS> i m pretty good thanks for asking <EOS>
2,2,<SOS> i m pretty good thanks for asking <EOS>,<SOS> no problem so how have you been <EOS>
3,3,<SOS> no problem so how have you been <EOS>,<SOS> i ve been great what about you <EOS>
4,4,<SOS> i ve been great what about you <EOS>,<SOS> i ve been good i m in school right now ...


In [39]:
max_len_lst = max(max(df['question'].apply(len)), max(df['answer'].apply(len)))

In [40]:
#padding
def padding(data):
    padding = []
    padded_seq = []
    for i in range(max_len_lst-len(data)):
        padding.append(vocab["<PAD>"])

    padded_seq = data + padding 
    # print(padded_seq)
    return padded_seq

# test = test.apply(padding)
test

0         [1499, 41, 20, 5, 176, 4, 34, 604, 41, 37, 552]
1       [4, 34, 604, 41, 37, 552, 4, 34, 160, 48, 244,...
2       [4, 34, 160, 48, 244, 29, 486, 33, 172, 25, 41...
3       [33, 172, 25, 41, 18, 5, 101, 4, 71, 101, 105,...
4       [4, 71, 101, 105, 13, 37, 5, 4, 71, 101, 48, 4...
                              ...                        
3720    [11, 10, 9, 48, 564, 81, 8, 10, 43, 134, 585, ...
3721                   [20, 5, 70, 2286, 28, 60, 30, 283]
3722    [28, 60, 30, 283, 5, 52, 1376, 75, 47, 70, 422...
3723    [5, 52, 1376, 75, 47, 70, 422, 218, 419, 8, 25...
3724    [32, 4, 14, 60, 30, 768, 53, 30, 70, 422, 257,...
Name: text, Length: 3725, dtype: object

In [41]:
#tokenizing
df["question"] = df["question"].apply(tokenize)
df["answer"] = df["answer"].apply(tokenize)

In [42]:
#padding
train_x = df["question"].apply(padding)
train_y = df["answer"].apply(padding)
train_x = list(train_x)
train_y = list(train_y)

In [43]:
#device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device : {device}")

device : cuda


In [44]:
#building dataset
class Chatdata(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, index):
       x = torch.tensor(self.x[index], dtype=torch.long)
       y = torch.tensor(self.y[index], dtype=torch.long)
       return x, y

In [45]:
#data loader
loader = DataLoader(dataset=Chatdata(train_x, train_y), batch_size=16, shuffle=True)

In [46]:
#imlpemeting transformer
#Self Attention ==> Positional Encoding ==> Multihead Attention
#Position-wise Feed-Forward Networks ==> Encoder-Decoder Architecure

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model//num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        
        attention_probability = torch.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_probability, V)
        return output
    
    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
    
    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
    
    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attention_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attention_output))
        return output


In [47]:
#FFN
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [48]:
#Postional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [49]:
#Encoder Layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attention_output = self.self_attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attention_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [50]:
#Decoder Layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.cross_attention = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, source_mask, target_mask):
        attention_output = self.self_attention(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attention_output))

        attention_output = self.cross_attention(x, encoder_output, encoder_output, source_mask)
        x = self.norm2(x + self.dropout(attention_output))

        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [51]:
#Complete Transformer
class Transformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length=500, dropout=0.1):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(source_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(target_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.fc = nn.Linear(d_model, target_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, source, target):
        source_mask = (source != 0).unsqueeze(1).unsqueeze(2)
        target_mask = (target != 0).unsqueeze(1).unsqueeze(2)

        seq_length = target.size(1)

        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        nopeak_mask = nopeak_mask.to(device)
        target_mask = target_mask & nopeak_mask
        target_mask = target_mask.to(device)
        return source_mask, target_mask
    
    def forward(self, source, target):
        source_mask, target_mask = self.generate_mask(source, target)

        # source_mask, target_mask = source_mask.to(device), target_mask.to(device)

        source_embed = self.dropout(self.positional_encoding(self.encoder_embedding(source)))
        target_embed = self.dropout(self.positional_encoding(self.decoder_embedding(target)))

        encoder_output = source_embed
        for encoder_layer in self.encoder_layers:
            encoder_output = encoder_layer(encoder_output, source_mask)

        decoder_output = target_embed
        for decoder_layer in self.decoder_layers:
            decoder_output = decoder_layer(decoder_output, encoder_output, source_mask, target_mask)

        output = self.fc(decoder_output)
        return output



In [52]:
vocab_size = len(vocab)
d_model = 128
num_heads = 8
num_layers = 4
d_ff = 256
max_seq_length = 500
dropout = 0.1
model = Transformer(vocab_size, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)
model.to(device)

Transformer(
  (encoder_embedding): Embedding(2459, 128)
  (decoder_embedding): Embedding(2459, 128)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-3): 4 x EncoderLayer(
      (self_attention): MultiHeadAttention(
        (W_q): Linear(in_features=128, out_features=128, bias=True)
        (W_k): Linear(in_features=128, out_features=128, bias=True)
        (W_v): Linear(in_features=128, out_features=128, bias=True)
        (W_o): Linear(in_features=128, out_features=128, bias=True)
      )
      (feed_forward): PositionWiseFeedForward(
        (fc1): Linear(in_features=128, out_features=256, bias=True)
        (fc2): Linear(in_features=256, out_features=128, bias=True)
        (relu): ReLU()
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0-3): 4 x DecoderLayer

In [53]:
#training loop
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
max_seq_length = 500

def train_step(model, x, y):
    model.train()

    x, y = x.to(device), y.to(device)

    dec_inp = y[:,:-1]
    target = y[:, 1:].contiguous().view(-1)

    logits = model(x, dec_inp)
    logits = logits.view(-1, logits.size(-1))

    loss = criterion(logits, target)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()

In [54]:
epochs = 50
for epoch in range(epochs):
    running_loss = 0.0

    for idx, (x, y) in enumerate(loader):
        
        loss = train_step(model, x, y)

        running_loss += loss

    print(f"Epoch: {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")
    torch.cuda.empty_cache()

Epoch: 1/50, Loss: 5.3332
Epoch: 2/50, Loss: 4.5513
Epoch: 3/50, Loss: 4.1689
Epoch: 4/50, Loss: 3.8577
Epoch: 5/50, Loss: 3.5935
Epoch: 6/50, Loss: 3.3470
Epoch: 7/50, Loss: 3.1199
Epoch: 8/50, Loss: 2.9081
Epoch: 9/50, Loss: 2.7224
Epoch: 10/50, Loss: 2.5473
Epoch: 11/50, Loss: 2.3863
Epoch: 12/50, Loss: 2.2455
Epoch: 13/50, Loss: 2.1331
Epoch: 14/50, Loss: 2.0169
Epoch: 15/50, Loss: 1.9265
Epoch: 16/50, Loss: 1.8376
Epoch: 17/50, Loss: 1.7775
Epoch: 18/50, Loss: 1.7056
Epoch: 19/50, Loss: 1.6501
Epoch: 20/50, Loss: 1.5955
Epoch: 21/50, Loss: 1.5545
Epoch: 22/50, Loss: 1.5139
Epoch: 23/50, Loss: 1.4671
Epoch: 24/50, Loss: 1.4425
Epoch: 25/50, Loss: 1.4044
Epoch: 26/50, Loss: 1.3732
Epoch: 27/50, Loss: 1.3359
Epoch: 28/50, Loss: 1.3154
Epoch: 29/50, Loss: 1.2853
Epoch: 30/50, Loss: 1.2583
Epoch: 31/50, Loss: 1.2356
Epoch: 32/50, Loss: 1.2079
Epoch: 33/50, Loss: 1.1839
Epoch: 34/50, Loss: 1.1621
Epoch: 35/50, Loss: 1.1428
Epoch: 36/50, Loss: 1.1117
Epoch: 37/50, Loss: 1.0891
Epoch: 38/

In [73]:

idx_to_word = {idx : word for word, idx in vocab.items() }


def infer(model, x, max_len=100):
    model.eval()
    # x = clean_data(x)
    x = tokenize(x)
    x = torch.tensor(x).to(device).unsqueeze(0)


    SOS = [vocab["<SOS>"]]

    for _ in range(max_len):
        dec_inp = torch.tensor(SOS, device=device).unsqueeze(0)
        
        with torch.no_grad():
            logits = model(x, dec_inp)[:, -1]

        next_token = torch.argmax(logits, dim=1).item()
        SOS.append(next_token)

        if next_token == vocab["<EOS>"]:
            break

    return " ".join(idx_to_word[i] for i in SOS if i not in (2, 3))


print(infer(model, "i m fine how about yourself"))


i m pretty good thanks for asking


In [ ]:
#Evaluation
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
model.load_state_dict(torch.load("final_bot.pth"))
model.eval()
reference = "i m good, how about you"
reference = reference.split()

predicted = infer(model, "i m fine how about yourself")
predicted = predicted.split()

bleu = sentence_bleu([reference], predicted, smoothing_function=SmoothingFunction().method2)
print("BlEU Score: ", bleu)

BlEU Score:  0.2283945119649991


C:\Users\supra\AppData\Local\Temp\ipykernel_3892\298587433.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("final_bot.pth"))


In [ ]:
# torch.save(model.state_dict(), "final_bot.pth")